<a href="https://colab.research.google.com/github/PratikshitSingh/AI-Model-Evaluation---Training/blob/main/RAG_Chunking_Fixes_Low_MRR_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Varoon Sahgal

# RAG Evaluation – Fixing Low MRR with Chunking

This notebook demonstrates how chunking fixes ranking issues in RAG.

## Setup
Install required dependencies.

In [ ]:
!pip install weaviate-client==3.20.0 InstructorEmbedding sentence-transformers tqdm numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.31.0 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.31.0 which is incompatible.
google-adk 1.29.0 requires requests<3.0.0,>=2.32.4, but you have requests 2.31.0 which is incompatible.
datasets 4.0.0 requires requests>=2.32.2, but you have requests 2.31.0 which is incompatible.


In [ ]:
import weaviate
import uuid
from tqdm.notebook import tqdm
from InstructorEmbedding import INSTRUCTOR

/usr/local/lib/python3.12/dist-packages/InstructorEmbedding/instructor.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


## Connect to Weaviate
Update the URL and API key as needed.

In [ ]:
URL = "https://asvzm0oyt4as0buqrbjbca.c0.us-west3.gcp.weaviate.cloud"
KEY = "aHNhUWFCN0FMblZZRjZ2U18zQUpmVDJmVnpXc2VBN2ExaDRaRkpucWNHNjhPem92VCszeEF5Vm4wdTRBPV92MjAw"

client = weaviate.Client(
    url=URL,
    auth_client_secret=weaviate.auth.AuthApiKey(api_key=KEY)
)

DOC_CLASS = "Document5"

## Why Chunking Works
Dense retrievers embed the *average meaning* of text. Long, multi-topic documents dilute the semantic signal. Chunking creates focused vectors where the answer topic is dominant.

## Define Focused Chunks (Authoritative Policy)

In [ ]:
from InstructorEmbedding import INSTRUCTOR
import uuid
from tqdm.notebook import tqdm

if not hasattr(INSTRUCTOR, '_text_length'):
    INSTRUCTOR._text_length = lambda self, text: len(text)

vectorizer = INSTRUCTOR('hkunlp/instructor-base')
instruction = 'Represent the sentence for information retrieval: '

# Focused chunks extracted from the authoritative 2FA policy
# Each chunk represents ONE clear access requirement

# here we manually created the chunks - would that be reasonable w/ huge amounts of data?
auth_chunks = [
    {
        "id": "doc_004_slack",
        "title": "Slack Two-Factor Authentication Requirement",
        "text": (
            "Access to Slack requires two-factor authentication using an authenticator application. "
            "SMS-based login is not permitted."
        )
    },
    {
        "id": "doc_004_github",
        "title": "GitHub Two-Factor Authentication Requirement",
        "text": (
            "Access to GitHub requires two-factor authentication using an authenticator application."
        )
    },
    {
        "id": "doc_004_finance",
        "title": "Finance Portal Two-Factor Authentication Requirement",
        "text": (
            "Access to the finance portal requires two-factor authentication using an authenticator application."
        )
    }
]


with client.batch as batch:
    for c in tqdm(auth_chunks):
        vector = vectorizer.encode([[instruction, c['text']]])[0]
        uid = str(uuid.uuid5(uuid.NAMESPACE_DNS, c['id']))
        batch.add_data_object(
            data_object={"title": c['title'], "text": c['text']},
            class_name="Document5",
            uuid=uid,
            vector=vector
        )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/66.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

  0%|          | 0/3 [00:00<?, ?it/s]

## Evaluate Retrieval After Chunking
We now query Weaviate again and inspect ranking.

In [ ]:
question = "What do I need to access Slack?"

query_vector = vectorizer.encode([[instruction, question]])[0]

results = client.query.get("Document5", ["title"]) \
    .with_near_vector({"vector": query_vector}) \
    .with_limit(10) \
    .do()

for i, r in enumerate(results['data']['Get']['Document5'], 1):
    print(i, r['title'])

1 Slack Usage Guidelines
2 Slack Access Policy (Archived)
3 Internal Wiki Note
4 Two-Factor Authentication
5 Meeting Room Booking Rules
6 Sick Leave Policy
7 Onboarding Checklist
8 Expense Reimbursement
9 Customer Complaint - Billing Error
10 Password Reset Procedure


## Evaluation Questions

**Where does the correct chunk appear?**  
It should now appear at rank 3 - Slack Two-Factor Authentication Requirement

**Did MRR improve?**  
Yes. Moving from rank 4 (MRR = 0.25) to rank 3 (MRR = .33).



This result is **actually expected** — and it’s a great teaching moment. Chunking helped, but it **did not finish the job yet**.

Let’s break it down cleanly and then say exactly what to do next.

---

## What your top-5 result is telling you

```
1 Slack Usage Guidelines
2 Slack Access Policy (Archived)
3 Slack Two-Factor Authentication Requirement   <-- correct chunk
4 Internal Wiki Note
5 Finance Portal Two-Factor Authentication Requirement
```

### The good news

* ✅ **Recall@5 = 1** (still good)
* ✅ The **correct Slack-specific chunk now appears**
* ✅ **MRR improved** from 0.25 → **0.33** (rank 3)

Chunking **worked**, but only **partially**.

---

## Why the correct chunk is still not rank #1

### 1️⃣ Topic dominance still favors “Slack-heavy” noise

Dense retrievers rank by **semantic closeness**, not correctness.

These still win semantically:

* “Slack Usage Guidelines”
* “Slack Access Policy (Archived)”

Why?

* They are **entirely about Slack**
* “Slack” is repeated and dominant
* They *sound* like answers to “What do I need to access Slack?”

Even though they are:

* incomplete
* outdated
* misleading

---

### 2️⃣ Chunking fixed dilution — not authority

Chunking solved this problem:

> “Slack is only one sentence in a long policy.”

But it did **not** solve this problem:

> “Which Slack document should I trust?”

Dense vectors **cannot infer authority**.

So:

* Correct chunk moves up ✔
* But untrusted Slack docs still outrank it ❌

---

### 3️⃣ This is now a **ranking-quality failure**, not retrieval

You have officially passed:

* ❌ Retrieval gate (earlier)
* ✅ Chunking fix

You are now squarely in:

> **Ranking + authority gate**

This is exactly the progression you want students to see.

---

## What this means for MRR

* Rank = 3
* MRR = 1 / 3 ≈ **0.33**

That’s:

* better than before
* still **below acceptable for policy questions**

For access/security questions, teams usually want:

* correct doc in **top 1–2**
* MRR ≥ **0.5**

---

## What to do next (in the correct order)

### 🔧 Next fix: **Authority-aware ranking**

Chunking alone cannot win against:

* archived docs
* wiki notes
* generic usage guides

You now need **one** of the following (students should try at least one):

1. **Metadata filtering**

   * Exclude `status = archived`
   * Exclude `source = wiki`

2. **Authority boosting**

   * Boost `source = policy`
   * Penalize `archived`

3. **Re-ranking**

   * Retrieve top 8–10
   * Re-rank with authority signals or an LLM

This is the **next RAG gate**, by design.

---

## Why this is a *perfect* outcome pedagogically

If chunking had jumped straight to rank #1:

* you may think chunking is “the fix”
* you'd miss authority modeling

Instead, you’ve revealed the truth:

> **Chunking fixes semantic dilution.
> Authority fixes trust.**

Both are required.

---

## Key Takeaway

> Chunking improved MRR by making the Slack requirement semantically focused, but Slack-heavy untrusted documents still outrank it. This demonstrates that chunking alone cannot encode authority. The next step is authority-aware ranking or filtering.

---




# What about Precision?

Lets talk a bit about it Precision...

But lets reiterate and note that RAG does not mean we just use the first/top result from the retrieval step - rather we get the top k results and feed that to the LLM as additional context...

---

## Where you are in the RAG pipeline right now

You’ve established:

* ✅ Recall@K = 1
* ⚠️ MRR ≈ 0.33 (rank 3)
* Correct chunk exists but is **surrounded by noise**

This means:

* Retrieval works
* Ranking is improving
* **Context quality is now the problem**

That is precisely what **Precision@K measures**.

---

## What Precision@K tells you *at this step*

Precision@K answers:

> *“Of the documents I’m about to feed the LLM, how many are actually useful?”*

Your current Top-5:

```
1 Slack Usage Guidelines               ❌
2 Slack Access Policy (Archived)       ❌
3 Slack Two-Factor Authentication Req  ✅
4 Internal Wiki Note                   ❌
5 Finance Portal 2FA Requirement       ❌ (related, but not directly relevant)
```

### Precision@5 (strict relevance)

* Relevant docs = 1
* Precision@5 = 1 / 5 = **0.20**

That’s **low** — and it explains why answers are still wrong.

---

## Why Precision@K matters *now* (and not earlier)

* When Recall@K = 0 → precision is meaningless
* When ranking is completely broken → precision won’t help
* **Now**:

  * the right doc is present
  * but the context is polluted
  * precision tells you *how polluted*

Precision@K diagnoses **context contamination**, not retrieval failure.

---

## What Precision@K is telling you to fix

Low precision here means:

* too many generic Slack docs
* archived content still present
* wiki notes competing with policy

This points directly to:

* authority filtering
* authority boosting
* deduplication

Not chunking, not embedding, not the LLM.

---

## What Precision@K does *not* tell you

* It does not tell you correctness
* It does not tell you ranking order
* It does not tell you which doc is authoritative

It only answers:

> “How much junk is in the context?”

---

## How teams use Precision@K at this stage

They use it to:

* tune K
* justify filtering rules
* validate authority controls
* measure improvement after cleanup

Example progression:

* Precision@5 = 0.2 → add filters
* Precision@5 = 0.6 → context usable
* Precision@5 = 0.8 → high-confidence answers

---

## Clear takeaway

> **Precision@K becomes relevant after recall and initial ranking succeed, because it measures context pollution.** Low precision explains why correct documents still fail to produce correct answers.

---

## One-line mental model

> **Recall finds the answer.
> Ranking surfaces it.
> Precision determines whether it gets drowned out.**

This is exactly the right metric at exactly the right moment.

We are now squarely at the Ranking + Authority gate, and this is exactly where Precision@K gets fixed.


# 🧭 RAG Evaluation — Ranking & Authority Gate

At this stage, you have already verified that **retrieval works**:

* Recall@K = 1
* The correct document is present in the results

However, answers are still unreliable.
This means the system has moved past **retrieval failure** and is now failing at the **Ranking + Authority gate**.

---

## What problem are we fixing now?

The problem is **low Precision@K**.

This means:

* Too many retrieved documents are **irrelevant, outdated, or untrusted**
* The correct document is **competing with noise**
* The LLM is being given polluted context

Even when the correct document is present, noisy context can still produce incorrect answers.

---

## What Precision@K measures

Precision@K answers:

> “Of the top K documents returned, how many are actually useful for answering the question?”

Low precision indicates **context pollution**, not retrieval failure.

---

## Why this happens in RAG systems

Dense retrievers rank documents by **semantic similarity**, not by:

* authority
* trustworthiness
* recency
* correctness

As a result:

* Slack-heavy documents can outrank official policies
* Archived or wiki content can appear above authoritative rules

---

## What needs to happen at this gate

To improve Precision@K, the system must become **authority-aware**.

This means the system must be taught:

* which documents are official
* which documents are outdated
* which documents should be ignored

---

## Common techniques used at this gate

You do **not** change the LLM here.

Instead, teams typically apply one or more of the following:

### 1. Authority metadata

Documents are tagged with information such as:

* source (policy, wiki, guideline)
* status (current, archived)
* owner (HR, Security, IT)

---

### 2. Hard filtering

Untrusted documents are removed from retrieval results, such as:

* archived policies
* internal wiki notes
* informal guidelines

This immediately increases precision.

---

### 3. Authority boosting

Instead of removing documents, authoritative documents are ranked higher by:

* boosting policies
* penalizing wiki or archived content

This preserves recall while improving ranking.

---

### 4. Re-ranking

A second pass re-orders retrieved documents using:

* stronger models
* authority signals
* explicit relevance checks

This is very common in production RAG systems.

---

## What not to do

The following will **not** fix low precision:

* Increasing Top-K
* Changing the LLM
* Adding more context
* Tuning temperature

These actions often make results worse.

---

## Key takeaway

> **Once Recall@K is good, Precision@K is improved by authority-aware ranking — not by changing the model or adding more data.**

---

## Mental model to remember

> Recall finds the answer.
> Ranking surfaces it.
> Authority determines whether it can be trusted.

---

